# 03 — XGBoost Training

**Purpose:** Train the demand forecasting model and compare it to the WMA baseline.

**Run after notebook 02.** Assumes `X_train`, `y_train`, `X_test`, `y_test`, `FEATURE_COLS`, and `wma_mae` (from notebook 01) are in memory.

**How XGBoost works (simplified):**
XGBoost builds hundreds of small decision trees. Each tree tries to correct the errors of the previous tree. After 300 trees, the model has learned complex patterns: *"when lag_1 is high AND it's a Monday AND last_30_avg is rising → predict higher than average."* No manual rules — it discovers them from data.

**`tree_method='hist'`** is critical for performance on large datasets — it bins feature values into histograms before splitting, which is ~10× faster than the default.

---

## Cell 1 — Imports and MLflow setup

MLflow is an experiment tracking tool. Think of it as a logbook: every training run records the hyperparameters used, the accuracy achieved, and saves the model artifact. You can then compare runs visually at `http://localhost:5001`.

**Why MLflow?** Without it, you'd have to remember "I tried max_depth=6 and got MAE=2.7, then tried max_depth=8 and got MAE=2.5" — MLflow records all of this automatically.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

# MLflow tracking — stored in ml/mlflow.db (SQLite, local)
ML_ROOT = Path.cwd().parent  # ml/ directory
mlflow.set_tracking_uri(f"sqlite:///{ML_ROOT}/mlflow.db")
mlflow.set_experiment("demand_forecasting")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment: demand_forecasting")
print("\nTo view runs: mlflow ui --backend-store-uri sqlite:///ml/mlflow.db --port 5001")

## Cell 2 — Define hyperparameters

These are the knobs that control how the model learns. The defaults below are well-tested starting points for retail demand forecasting:

| Parameter | Value | What it does |
|---|---|---|
| `n_estimators` | 300 | Number of trees. More = better but slower. Start here. |
| `max_depth` | 6 | Max depth per tree. 6 is a good balance — deep enough to learn patterns, shallow enough to not overfit. |
| `learning_rate` | 0.05 | How much each tree corrects. Smaller = slower but more stable. |
| `subsample` | 0.8 | Train each tree on 80% of rows (random). Reduces overfitting. |
| `colsample_bytree` | 0.8 | Use 80% of features per tree. Reduces overfitting. |
| `min_child_weight` | 5 | Min samples per leaf. Higher = less overfit on sparse products. |
| `tree_method` | hist | Fast histogram-based splitting (required for large datasets). |

In [ ]:
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}
print("Hyperparameters:", XGB_PARAMS)

## Cell 3 — Train the model

This is the actual training step. On ~28M rows (4.6M × 6 of 7 horizons in train), expect 3–8 minutes on a modern CPU.

**`eval_set`** passes the test set during training so XGBoost can print validation loss at each tree. This lets you see if the model is still improving or has plateaued — useful for tuning `n_estimators`.

**`verbose=50`** prints evaluation every 50 trees. If the test RMSE stops decreasing and starts increasing, the model is overfitting — you'd want fewer trees.

In [ ]:
# Assumes X_train, y_train, X_test, y_test are in memory from notebook 02
model = xgb.XGBRegressor(**XGB_PARAMS)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print("\nTraining complete.")

## Cell 4 — Evaluate on test set

Now we measure how well the model does on the held-out 30 days it has never seen.

**MAE** = Mean Absolute Error = average |actual - predicted| across all (product, date, horizon) rows

**RMSE** = Root Mean Square Error = like MAE but penalises large errors more. A prediction off by 10 units hurts 4× more than one off by 5 units.

**Per-horizon MAE** is also interesting: horizon=1 predictions should be more accurate than horizon=7 (further future = more uncertainty).

In [ ]:
preds = model.predict(X_test)
test_mae  = mean_absolute_error(y_test, preds)
test_rmse = float(np.sqrt(mean_squared_error(y_test, preds)))

print(f"Model  — MAE: {test_mae:.4f}  |  RMSE: {test_rmse:.4f}")

# Per-horizon breakdown
test_set_eval = X_test.copy()
test_set_eval["actual"] = y_test.values
test_set_eval["predicted"] = preds
test_set_eval["abs_error"] = np.abs(test_set_eval["actual"] - test_set_eval["predicted"])

print("\nMAE per horizon:")
print(test_set_eval.groupby("horizon_days")["abs_error"].mean().round(4).to_string())

## Cell 5 — Compare to WMA baseline

The moment of truth: is the model better than the current system?

We recompute WMA on the same test rows (not just the 1-day rows — all horizons see the same WMA prediction since WMA doesn't account for horizon).

In [ ]:
# Recompute WMA on the same test rows used above
# test_set must be the DataFrame from notebook 02 (not just X_test)
wma_pred_test = (
    0.6 * test_set["last_7_day_avg"].fillna(0)
    + 0.3 * test_set["last_30_day_avg"].fillna(0)
    + 0.1 * test_set["last_60_day_avg"].fillna(0)
)
wma_mae_test  = mean_absolute_error(y_test, wma_pred_test)
wma_rmse_test = float(np.sqrt(mean_squared_error(y_test, wma_pred_test)))
improvement   = (wma_mae_test - test_mae) / wma_mae_test * 100

print("=" * 40)
print(f"WMA baseline  — MAE: {wma_mae_test:.4f}  RMSE: {wma_rmse_test:.4f}")
print(f"XGBoost model — MAE: {test_mae:.4f}  RMSE: {test_rmse:.4f}")
print(f"Improvement:  {improvement:+.1f}%")
print("=" * 40)

## Cell 6 — Log to MLflow

Record this run so you can compare it to future experiments. After logging, open the MLflow UI:
```bash
mlflow ui --backend-store-uri sqlite:///ml/mlflow.db --port 5001
```
Then visit `http://localhost:5001` to see all runs, compare metrics, and inspect parameters.

In [ ]:
with mlflow.start_run(run_name="notebook_baseline") as run:
    mlflow.log_params(XGB_PARAMS)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("wma_mae", wma_mae_test)
    mlflow.log_metric("wma_rmse", wma_rmse_test)
    mlflow.log_metric("improvement_pct", improvement)
    mlflow.log_metric("train_rows", len(X_train))
    mlflow.log_metric("test_rows", len(X_test))
    mlflow.xgboost.log_model(model, "model")
    run_id = run.info.run_id

print(f"Run logged. Run ID: {run_id}")
print(f"View at: http://localhost:5001")

## Cell 7 — Feature importance (built-in)

XGBoost has a built-in feature importance metric (`weight` = how often a feature is used in splits). This gives a quick sense of which features matter most.

**Expected:** `lag_1_qty` and `last_7_day_avg` should dominate. `horizon_days` should also rank high — the model heavily uses it to adjust predictions for different forecast distances.

In [ ]:
import matplotlib.pyplot as plt

importance = model.get_booster().get_fscore()
imp_df = pd.DataFrame(list(importance.items()), columns=["feature", "score"]).sort_values("score", ascending=False)
print(imp_df.to_string(index=False))

imp_df.plot.barh(x="feature", y="score", figsize=(8, 5), legend=False)
plt.title("XGBoost feature importance (split count)")
plt.tight_layout()
plt.show()